# Research-Level Benchmark: Backbone Swapping + ONNX/PTQ

This notebook treats **backbone swapping** and **post-training quantization (PTQ)** as two equally important experimental dimensions.

## Experimental questions

**Part I — Backbone Swapping / Transfer Learning**
- Can the same classification head and training protocol be used with different backbones?
- How do **ResNet-18, ResNet-50, ViT-B/16, Faster R-CNN ResNet-50 FPN, and a custom residual CNN** compare?
- What is the trade-off between accuracy, macro-F1, weighted-F1, parameter count, model size, training time, inference latency, throughput, and peak memory?

**Part II — ONNX + PTQ**
- For every selected backbone, how does deployment change after:
  1. PyTorch FP32
  2. ONNX FP32
  3. ONNX Dynamic INT8
  4. ONNX Static INT8
- How much accuracy/F1 is lost or retained?
- How much latency, model size, and memory improve?
- Does the effect of PTQ depend on the backbone?

The key research design is therefore:

> **Backbone × Deployment Artifact**

This prevents PTQ from being evaluated on only one architecture and prevents backbone swapping from being treated as a superficial implementation detail.

> **Important:** Quantization is a deployment transformation. A quantized model should not be described as "more accurate" merely because its measured accuracy fluctuates upward on a finite test subset.

In [ ]:
!pip install -q onnx onnxruntime onnxscript psutil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 21.3 MB/s eta 0:00:00


In [ ]:
import os, time, random, gc, json, math, platform
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import (
    resnet18, resnet50, ResNet18_Weights, ResNet50_Weights,
    vit_b_16, ViT_B_16_Weights
)
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FasterRCNN_ResNet50_FPN_Weights

import onnxruntime as ort
from onnxruntime.quantization import (
    quantize_dynamic, quantize_static, CalibrationDataReader,
    QuantType, QuantFormat, quant_pre_process
)

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

In [ ]:
SEED = 42
DATASET_NAME = "CIFAR10"


EPOCHS = 30
BATCH_SIZE = 16
NUM_WORKERS = 2
MAX_EVAL_SAMPLES = 1000
CALIBRATION_BATCHES = 10
LATENCY_WARMUP = 10
LATENCY_RUNS = 50

BACKBONES = [
    "resnet18",
    "resnet50",
    "vit_b_16",
    "faster_r_cnn",
    "customcnn",
]

BACKBONE_CONFIGS = {
    "resnet18": {
        "lr_backbone": 3e-4, "lr_head": 1e-3, "weight_decay": 1e-4,
        "warmup_epochs": 0, "freeze_backbone": False,
    },
    "resnet50": {
        "lr_backbone": 3e-4, "lr_head": 1e-3, "weight_decay": 1e-4,
        "warmup_epochs": 0, "freeze_backbone": False,
    },
    "vit_b_16": {
        "lr_backbone": 6e-5, "lr_head": 1e-3, "weight_decay": 5e-2,
        "warmup_epochs": 3, "freeze_backbone": False,
    },
    "faster_r_cnn": {
        "lr_backbone": 3e-4, "lr_head": 1e-3, "weight_decay": 1e-4,
        "warmup_epochs": 2, "freeze_backbone": False,
    },
    "customcnn": {
        "lr_backbone": 3e-3, "lr_head": 3e-3, "weight_decay": 5e-5,
        "warmup_epochs": 0, "freeze_backbone": False,
    },
}

RESULTS_DIR = Path("research_results")
RESULTS_DIR.mkdir(exist_ok=True)

print("PyTorch:", torch.__version__)
print("ONNX Runtime:", ort.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.11.0+cu128
ONNX Runtime: 1.29.0
Device: cuda


In [ ]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def reset_peak_memory():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

def peak_gpu_memory_mb():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / (1024**2)
    return np.nan

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def file_size_mb(path):
    return os.path.getsize(path) / (1024**2)

print("Device:", device)

Device: cuda


## Part I — Backbone Swapping and Transfer Learning

All candidate backbones feed the **same classification objective**. The experiment records both predictive quality and computational cost.

The backbone factory explicitly supports:

- `resnet18`
- `resnet50`
- `vit_b_16`
- `faster_r_cnn`
- `customcnn`

The comparison is not simply "which model has the highest accuracy"; it is a multi-objective benchmark of **accuracy versus computational cost**.

Note: ResNet-18/50, ViT-B/16, and the Faster R-CNN backbone use ImageNet-pretrained weights. The custom CNN is trained from random initialization. This experiment therefore compares pretrained transfer learning vs. a from-scratch custom architecture, not five architectures under identical initialization conditions — interpret backbone rankings accordingly.

In [ ]:
class Residual(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch)
        )
        self.skip = nn.Identity()
        if stride != 1 or in_ch != out_ch:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_ch)
            )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.conv(x) + self.skip(x))

class CustomCNNBackbone(nn.Module):
    def __init__(self, input_channels=3):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(input_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True)
        )
        self.res1 = Residual(16, 32)
        self.res2 = Residual(32, 32)
        self.pool1 = nn.MaxPool2d(2)
        self.res3 = Residual(32, 64)

    def forward(self, x):
        return self.res3(self.pool1(self.res2(self.res1(self.conv1(x)))))

class ViTBackboneWrapper(nn.Module):
    def __init__(self, vit_model):
        super().__init__()
        self.conv_proj = vit_model.conv_proj
        self.class_token = vit_model.class_token
        self.encoder = vit_model.encoder
        self.patch_size = vit_model.patch_size
        self.hidden_dim = vit_model.hidden_dim

    def forward(self, x):
        n, c, h, w = x.shape
        p = self.patch_size
        n_h, n_w = h // p, w // p

        x = self.conv_proj(x)
        x = x.reshape(n, self.hidden_dim, n_h * n_w)
        x = x.permute(0, 2, 1)

        class_token = self.class_token.expand(n, -1, -1)
        x = torch.cat([class_token, x], dim=1)
        x = self.encoder(x)
        return x

def get_backbone(backbone_name, pretrained=True):
    name = backbone_name.lower()

    if name == "resnet18":
        model = resnet18(weights=ResNet18_Weights.DEFAULT if pretrained else None)
        return nn.Sequential(*list(model.children())[:-2]), model.fc.in_features, "cnn"

    if name == "resnet50":
        model = resnet50(weights=ResNet50_Weights.DEFAULT if pretrained else None)
        return nn.Sequential(*list(model.children())[:-2]), model.fc.in_features, "cnn"

    if name == "faster_r_cnn":
        rcnn = fasterrcnn_resnet50_fpn(
            weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT if pretrained else None
        )
        return rcnn.backbone, 256, "fpn"

    if name == "vit_b_16":
        model = vit_b_16(weights=ViT_B_16_Weights.DEFAULT if pretrained else None)
        return ViTBackboneWrapper(model), model.hidden_dim, "vit"

    if name == "customcnn":
        return CustomCNNBackbone(), 64, "cnn"

    raise ValueError(f"Unsupported backbone: {backbone_name}")

In [ ]:
class CustomModel(nn.Module):
    def __init__(self, backbone_name, num_classes, pretrained=True):
        super().__init__()
        self.backbone_name = backbone_name.lower()
        self.backbone, in_features, feature_type = get_backbone(
            self.backbone_name, pretrained
        )
        self.feature_type = feature_type

        if feature_type in ("cnn", "fpn"):
              self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        self.flatten = nn.Flatten()
        self.classifier = nn.Linear(in_features, num_classes)

    def forward(self, x):
        features = self.backbone(x)

        if isinstance(features, (dict, OrderedDict)):
            features = features["0"]

        if self.feature_type == "vit":
            features = features[:, 0]
            return self.classifier(features)

        features = self.avgpool(features)
        features = self.flatten(features)
        return self.classifier(features)

def dataset_decider(name=DATASET_NAME, batch_size=BATCH_SIZE, val_fraction=0.1):
    train_transform = transforms.Compose([
        transforms.RandomRotation(10),
        transforms.RandomHorizontalFlip(),
        transforms.Resize(256),
        transforms.RandomResizedCrop(224),
        transforms.ColorJitter(0.1, 0.1, 0.1),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    eval_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    name = name.upper()

    if name == "CIFAR10":
        train_ds_full = datasets.CIFAR10(root="data", train=True, download=True, transform=train_transform)
        val_ds_full = datasets.CIFAR10(root="data", train=True, download=True, transform=eval_transform)
        test_ds = datasets.CIFAR10(root="data", train=False, download=True, transform=eval_transform)
    elif name == "STL10":
        train_ds_full = datasets.STL10(root="data", split="train", download=True, transform=train_transform)
        val_ds_full = datasets.STL10(root="data", split="train", download=True, transform=eval_transform)
        test_ds = datasets.STL10(root="data", split="test", download=True, transform=eval_transform)
    else:
        raise ValueError(name)

    num_classes = len(train_ds_full.classes)

    # Carve validation split out of TRAIN, not test — checkpoint selection must never see the test set.
    rng = np.random.default_rng(SEED)
    n_total = len(train_ds_full)
    perm = rng.permutation(n_total)
    n_val = int(n_total * val_fraction)
    val_indices = perm[:n_val].tolist()
    train_indices = perm[n_val:].tolist()

    train_ds = Subset(train_ds_full, train_indices)
    val_ds = Subset(val_ds_full, val_indices)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
    )

    if MAX_EVAL_SAMPLES is not None:
        test_indices = rng.choice(
            len(test_ds), size=min(MAX_EVAL_SAMPLES, len(test_ds)), replace=False
        ).tolist()
        test_ds = Subset(test_ds, test_indices)

    eval_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
    )

    return train_loader, val_loader, eval_loader, num_classes

train_loader, val_loader, eval_loader, num_classes = dataset_decider()
class_names = datasets.CIFAR10(root="data", train=False).classes if DATASET_NAME.upper() == "CIFAR10" else None
print("Classes:", num_classes, "Evaluation samples:", len(eval_loader.dataset))

100%|██████████| 170M/170M [23:19<00:00, 122kB/s]


Classes: 10 Evaluation samples: 1000


In [ ]:
calibration_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

calibration_ds = datasets.CIFAR10(
    root="data", train=True, download=True, transform=calibration_transform
)

calibration_loader = DataLoader(
    calibration_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=False
)

In [ ]:
def build_optimizer_and_scheduler(model, cfg, steps_per_epoch, epochs):
    param_groups = [
        {"params": model.backbone.parameters(), "lr": cfg["lr_backbone"]},
        {"params": model.classifier.parameters(), "lr": cfg["lr_head"]},
    ]
    optimizer = optim.AdamW(param_groups, weight_decay=cfg["weight_decay"])

    warmup_steps = cfg["warmup_epochs"] * steps_per_epoch
    total_steps = epochs * steps_per_epoch

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    return optimizer, scheduler

In [ ]:
class Trainer:
    def __init__(self, model, train_loader, eval_loader, config):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.eval_loader = eval_loader
        self.loss_fn = nn.CrossEntropyLoss()

        if config.get("freeze_backbone", False):
            for p in self.model.backbone.parameters():
                p.requires_grad = False

        steps_per_epoch = len(train_loader)
        self.optimizer, self.scheduler = build_optimizer_and_scheduler(
            self.model, config, steps_per_epoch, config.get("epochs", EPOCHS)
        )

        amp_enabled = bool(config.get("use_amp", True) and torch.cuda.is_available())
        self.amp_enabled = amp_enabled
        self.scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
        self.grad_clip = config.get("use_grad_clip", True)

    def train_one_epoch(self):
        self.model.train()
        total_loss = 0.0
        start = time.perf_counter()

        for x, y in self.train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            self.optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(
                device_type="cuda",
                enabled=self.amp_enabled
            ):
                logits = self.model(x)
                loss = self.loss_fn(logits, y)

            self.scaler.scale(loss).backward()

            if self.grad_clip:
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)

            self.scaler.step(self.optimizer)
            self.scaler.update()
            self.scheduler.step()
            total_loss += loss.item()

        return total_loss / len(self.train_loader), time.perf_counter() - start

    @torch.no_grad()
    def evaluate(self):
        self.model.eval()
        losses, preds, labels = [], [], []
        start = time.perf_counter()

        for x, y in self.eval_loader:
            x, y = x.to(device), y.to(device)
            logits = self.model(x)
            losses.append(self.loss_fn(logits, y).item())
            preds.append(logits.argmax(1).cpu().numpy())
            labels.append(y.cpu().numpy())

        elapsed = time.perf_counter() - start
        preds = np.concatenate(preds)
        labels = np.concatenate(labels)

        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, preds, average="macro", zero_division=0
        )
        w_precision, w_recall, w_f1, _ = precision_recall_fscore_support(
            labels, preds, average="weighted", zero_division=0
        )

        return {
            "loss": float(np.mean(losses)),
            "accuracy": accuracy_score(labels, preds),
            "macro_precision": precision,
            "macro_recall": recall,
            "macro_f1": f1,
            "weighted_f1": w_f1,
            "eval_time_s": elapsed,
            "preds": preds,
            "labels": labels,
        }

    def fit(self, epochs):
        history = []
        best_acc = -1

        for epoch in range(epochs):
            train_loss, train_time = self.train_one_epoch()
            metrics = self.evaluate()

            row = {
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "train_time_s": train_time,
                **{k: v for k, v in metrics.items() if k not in ["preds", "labels"]}
            }
            history.append(row)

            print(
                f"Epoch {epoch+1}/{epochs} | "
                f"Loss {train_loss:.4f} | "
                f"Acc {metrics['accuracy']*100:.2f}% | "
                f"Macro-F1 {metrics['macro_f1']:.4f} | "
                f"Time {train_time:.1f}s"
            )

            if metrics["accuracy"] > best_acc:
                best_acc = metrics["accuracy"]
                torch.save(
                    self.model.state_dict(),
                    RESULTS_DIR / f"best_{self.model.backbone_name}.pth"
                )

        return pd.DataFrame(history), metrics

In [ ]:
@torch.no_grad()
def benchmark_torch_latency(model, loader, warmup=LATENCY_WARMUP, runs=LATENCY_RUNS):
    model.eval()
    batch = next(iter(loader))[0].to(device)

    # Warm-up
    for _ in range(warmup):
        _ = model(batch)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    timings = []
    for _ in range(runs):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        _ = model(batch)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        timings.append(time.perf_counter() - t0)

    return {
        "mean_batch_latency_ms": np.mean(timings) * 1000,
        "median_batch_latency_ms": np.median(timings) * 1000,
        "p95_batch_latency_ms": np.percentile(timings, 95) * 1000,
        "std_batch_latency_ms": np.std(timings) * 1000,
        "throughput_images_s": batch.shape[0] / np.mean(timings),
    }

def evaluate_model(model, loader):
    model.eval()
    preds, labels = [], []

    with torch.no_grad():
        for x, y in loader:
            logits = model(x.to(device))
            preds.append(logits.argmax(1).cpu().numpy())
            labels.append(y.numpy())

    preds = np.concatenate(preds)
    labels = np.concatenate(labels)

    p, r, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    wp, wr, wf1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_precision": p,
        "macro_recall": r,
        "macro_f1": f1,
        "weighted_f1": wf1,
        "preds": preds,
        "labels": labels,
    }

### Backbone experiment protocol

For every backbone:

1. Load the same pretrained/initialized backbone.
2. Attach the same classification objective.
3. Train with the same optimizer/scheduler policy.
4. Evaluate on the same test subset.
5. Measure:
   - Accuracy
   - Macro precision / recall / F1
   - Weighted F1
   - Parameter count
   - Trainable parameter count
   - PyTorch checkpoint size
   - Evaluation latency
   - Throughput
   - Latency variability
   - Peak GPU memory when CUDA is available

This creates the baseline against which the ONNX/PTQ experiments are compared.

In [ ]:
backbone_results = []
backbone_histories = {}
trained_models = {}

for backbone_name in BACKBONES:
    print("\n" + "="*80)
    print("BACKBONE:", backbone_name)
    print("="*80)

    clear_memory()
    reset_peak_memory()
    set_seed(SEED)

    t0 = time.perf_counter()
    model = CustomModel(backbone_name, num_classes, pretrained=True)
    construction_time = time.perf_counter() - t0

    total_params, trainable_params = count_parameters(model)

    config = {
        "use_amp": True,
        "use_grad_clip": True,
        "epochs": EPOCHS,
        **BACKBONE_CONFIGS[backbone_name],
    }

    trainer = Trainer(model, train_loader, val_loader, config)
    train_start = time.perf_counter()
    history, final_metrics = trainer.fit(EPOCHS)
    total_train_time = time.perf_counter() - train_start

    # Reload best checkpoint
    ckpt = RESULTS_DIR / f"best_{backbone_name}.pth"
    model.load_state_dict(torch.load(ckpt, map_location=device))

    reset_peak_memory()
    torch_metrics = evaluate_model(model, eval_loader)
    latency_metrics = benchmark_torch_latency(model, eval_loader)

    checkpoint_mb = file_size_mb(ckpt)

    record = {
        "Backbone": backbone_name,
        "Parameters_M": total_params / 1e6,
        "Trainable_M": trainable_params / 1e6,
        "Checkpoint_MB": checkpoint_mb,
        "Construction_Time_s": construction_time,
        "Training_Time_s": total_train_time,
        "Accuracy": torch_metrics["accuracy"],
        "Macro_Precision": torch_metrics["macro_precision"],
        "Macro_Recall": torch_metrics["macro_recall"],
        "Macro_F1": torch_metrics["macro_f1"],
        "Weighted_F1": torch_metrics["weighted_f1"],
        **latency_metrics,
        "Peak_GPU_Memory_MB": peak_gpu_memory_mb(),
    }

    backbone_results.append(record)
    backbone_histories[backbone_name] = history
    trained_models[backbone_name] = model.cpu()

    print(pd.DataFrame([record]).T)

backbone_df = pd.DataFrame(backbone_results).sort_values("Accuracy", ascending=False)
display(backbone_df.round(4))
backbone_df.to_csv(RESULTS_DIR / "backbone_comparison.csv", index=False)


BACKBONE: resnet18
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 179MB/s]
/tmp/ipykernel_978/2957000679.py:46: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  self.scheduler.step()


Epoch 1/30 | Loss 1.2238 | Acc 81.88% | Macro-F1 0.8174 | Time 229.0s
Epoch 2/30 | Loss 0.9655 | Acc 85.36% | Macro-F1 0.8517 | Time 226.9s
Epoch 3/30 | Loss 0.8646 | Acc 86.52% | Macro-F1 0.8639 | Time 227.0s
Epoch 4/30 | Loss 0.7939 | Acc 86.36% | Macro-F1 0.8635 | Time 225.3s
Epoch 5/30 | Loss 0.7383 | Acc 89.18% | Macro-F1 0.8914 | Time 224.2s
Epoch 6/30 | Loss 0.7040 | Acc 88.52% | Macro-F1 0.8842 | Time 225.6s
Epoch 7/30 | Loss 0.6718 | Acc 89.34% | Macro-F1 0.8924 | Time 226.4s
Epoch 8/30 | Loss 0.6331 | Acc 91.08% | Macro-F1 0.9099 | Time 224.3s
Epoch 9/30 | Loss 0.6021 | Acc 90.70% | Macro-F1 0.9059 | Time 233.2s
Epoch 10/30 | Loss 0.5771 | Acc 91.98% | Macro-F1 0.9187 | Time 228.8s
Epoch 11/30 | Loss 0.5537 | Acc 92.48% | Macro-F1 0.9245 | Time 226.2s
Epoch 12/30 | Loss 0.5256 | Acc 92.78% | Macro-F1 0.9272 | Time 229.4s
Epoch 13/30 | Loss 0.5051 | Acc 93.10% | Macro-F1 0.9305 | Time 231.9s
Epoch 14/30 | Loss 0.4824 | Acc 93.10% | Macro-F1 0.9306 | Time 227.1s
Epoch 15/30 | L

In [ ]:
# Backbone accuracy and F1 comparison
fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(backbone_df))
width = 0.25

ax.bar(x - width, backbone_df["Accuracy"], width, label="Accuracy")
ax.bar(x, backbone_df["Macro_F1"], width, label="Macro F1")
ax.bar(x + width, backbone_df["Weighted_F1"], width, label="Weighted F1")

ax.set_xticks(x)
ax.set_xticklabels(backbone_df["Backbone"], rotation=30, ha="right")
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Backbone Swapping — Predictive Performance")
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

# Accuracy vs parameter count
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(backbone_df["Parameters_M"], backbone_df["Accuracy"], s=100)
for _, row in backbone_df.iterrows():
    ax.annotate(row["Backbone"], (row["Parameters_M"], row["Accuracy"]))
ax.set_xlabel("Parameters (M)")
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy–Model Capacity Trade-off")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# Latency and throughput
fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(backbone_df["Backbone"], backbone_df["mean_batch_latency_ms"])
ax.set_ylabel("Mean batch latency (ms)")
ax.set_title("Backbone Inference Latency")
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

display(
    backbone_df[
        ["Backbone", "Accuracy", "Macro_F1", "Parameters_M",
         "Checkpoint_MB", "mean_batch_latency_ms", "throughput_images_s"]
    ].round(4)
)

## Part II — ONNX Export and PTQ

The second experiment is run **per backbone**, rather than only on the best-performing architecture.

For each backbone the deployment ladder is:

**PyTorch FP32 → ONNX FP32 → ONNX Dynamic INT8 → ONNX Static INT8**

The same evaluation subset and preprocessing are used for all artifacts.

For each artifact we measure:

- Accuracy
- Macro precision / recall / F1
- Weighted F1
- Mean latency
- Median latency
- P95 latency
- Throughput
- Latency standard deviation
- Serialized model size
- Compression ratio versus PyTorch FP32
- Accuracy delta versus PyTorch FP32
- F1 delta versus PyTorch FP32
- Latency speedup versus PyTorch FP32

This allows two levels of analysis:

1. **Within-backbone:** Does quantization improve deployment efficiency?
2. **Across-backbone:** Which architecture remains the best under a deployment constraint?

In [ ]:
class CalibrationReader(CalibrationDataReader):
    def __init__(self, dataloader, input_name="images", num_batches=10):
        self.input_name = input_name
        self.iterator = iter(dataloader)
        self.num_batches = num_batches
        self.count = 0

    def get_next(self):
        if self.count >= self.num_batches:
            return None
        try:
            x, _ = next(self.iterator)
            self.count += 1
            return {self.input_name: x.numpy().astype(np.float32)}
        except StopIteration:
            return None

def export_onnx(model, dummy_input, path):
    model = model.eval().cpu()
    torch.onnx.export(
        model,
        dummy_input.cpu(),
        str(path),
        input_names=["images"],
        output_names=["logits"],
        dynamic_axes={
            "images": {0: "batch_size"},
            "logits": {0: "batch_size"}
        },
        opset_version=18,
        do_constant_folding=True,
        dynamo=False
    )

def create_dynamic_int8(fp32_path, output_path):
    quantize_dynamic(
        str(fp32_path),
        str(output_path),
        weight_type=QuantType.QUInt8
    )

def create_static_int8(fp32_path, output_path, calibration_loader):
    preprocessed_path = str(fp32_path).replace(
        ".onnx", "_preprocessed.onnx"
    )

    quant_pre_process(
        str(fp32_path),
        preprocessed_path,
        skip_symbolic_shape=False
    )

    reader = CalibrationReader(
        calibration_loader,
        input_name="images",
        num_batches=CALIBRATION_BATCHES
    )

    quantize_static(
        preprocessed_path,
        str(output_path),
        calibration_data_reader=reader,
        quant_format=QuantFormat.QDQ,
        activation_type=QuantType.QUInt8,
        weight_type=QuantType.QUInt8
    )

def make_ort_session(path):
    return ort.InferenceSession(
        str(path),
        providers=["CPUExecutionProvider"]
    )

def ort_predict(session, loader):
    input_name = session.get_inputs()[0].name
    preds, labels = [], []

    for x, y in loader:
        logits = session.run(
            None,
            {input_name: x.numpy().astype(np.float32)}
        )[0]
        preds.append(np.argmax(logits, axis=1))
        labels.append(y.numpy())

    return np.concatenate(preds), np.concatenate(labels)

def benchmark_onnx_latency(session, loader, warmup=LATENCY_WARMUP, runs=LATENCY_RUNS):
    x, _ = next(iter(loader))
    x_np = x.numpy().astype(np.float32)
    input_name = session.get_inputs()[0].name

    for _ in range(warmup):
        session.run(None, {input_name: x_np})

    timings = []
    for _ in range(runs):
        t0 = time.perf_counter()
        session.run(None, {input_name: x_np})
        timings.append(time.perf_counter() - t0)

    timings = np.asarray(timings)
    return {
        "mean_batch_latency_ms": timings.mean() * 1000,
        "median_batch_latency_ms": np.median(timings) * 1000,
        "p95_batch_latency_ms": np.percentile(timings, 95) * 1000,
        "std_batch_latency_ms": timings.std() * 1000,
        "throughput_images_s": x_np.shape[0] / timings.mean(),
    }

def classification_metrics_from_predictions(labels, preds):
    p, r, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    wp, wr, wf1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    return {
        "Accuracy": accuracy_score(labels, preds),
        "Macro_Precision": p,
        "Macro_Recall": r,
        "Macro_F1": f1,
        "Weighted_F1": wf1,
    }

In [ ]:
def _nan_artifact_record(backbone_name, artifact_name):
    return {
        "Backbone": backbone_name,
        "Artifact": artifact_name,
        "Path": "",
        "Size_MB": np.nan,
        "Accuracy": np.nan,
        "Macro_Precision": np.nan,
        "Macro_Recall": np.nan,
        "Macro_F1": np.nan,
        "Weighted_F1": np.nan,
        "Mean_Latency_ms": np.nan,
        "Median_Latency_ms": np.nan,
        "P95_Latency_ms": np.nan,
        "Latency_STD_ms": np.nan,
        "Throughput_images_s": np.nan,
    }

deployment_results = []

for backbone_name, model in trained_models.items():
    print("\n" + "#"*90)
    print("DEPLOYMENT BENCHMARK:", backbone_name)
    print("#"*90)

    backbone_dir = RESULTS_DIR / backbone_name
    backbone_dir.mkdir(exist_ok=True)

    model = model.eval().cpu()

    dummy_input = torch.randn(BATCH_SIZE, 3, 224, 224)

    # Reference: PyTorch FP32 CPU
    torch_model_path = backbone_dir / "model_pytorch_fp32.pth"
    torch.save(model.state_dict(), torch_model_path)

    x0, y0 = next(iter(eval_loader))
    # Warmup
    with torch.no_grad():
        for _ in range(LATENCY_WARMUP):
            _ = model(x0)

    timings = []
    with torch.no_grad():
        for _ in range(LATENCY_RUNS):
            t0 = time.perf_counter()
            logits = model(x0)
            timings.append(time.perf_counter() - t0)

    torch_preds, torch_labels = [], []
    with torch.no_grad():
        for x, y in eval_loader:
            torch_preds.append(model(x).argmax(1).numpy())
            torch_labels.append(y.numpy())

    torch_preds = np.concatenate(torch_preds)
    torch_labels = np.concatenate(torch_labels)

    torch_metrics = classification_metrics_from_predictions(
        torch_labels, torch_preds
    )

    artifacts = [{
        "Backbone": backbone_name,
        "Artifact": "PyTorch FP32",
        "Path": str(torch_model_path),
        "Size_MB": file_size_mb(torch_model_path),
        **torch_metrics,
        "Mean_Latency_ms": np.mean(timings) * 1000,
        "Median_Latency_ms": np.median(timings) * 1000,
        "P95_Latency_ms": np.percentile(timings, 95) * 1000,
        "Latency_STD_ms": np.std(timings) * 1000,
        "Throughput_images_s": x0.shape[0] / np.mean(timings),
    }]

    # ONNX FP32
    fp32_path = backbone_dir / "model_fp32.onnx"
    try:
        export_onnx(model, dummy_input, fp32_path)

        session = make_ort_session(fp32_path)
        preds, labels = ort_predict(session, eval_loader)
        metrics = classification_metrics_from_predictions(labels, preds)
        latency = benchmark_onnx_latency(session, eval_loader)

        artifacts.append({
            "Backbone": backbone_name,
            "Artifact": "ONNX FP32",
            "Path": str(fp32_path),
            "Size_MB": file_size_mb(fp32_path),
            **metrics,
            "Mean_Latency_ms": latency["mean_batch_latency_ms"],
            "Median_Latency_ms": latency["median_batch_latency_ms"],
            "P95_Latency_ms": latency["p95_batch_latency_ms"],
            "Latency_STD_ms": latency["std_batch_latency_ms"],
            "Throughput_images_s": latency["throughput_images_s"],
        })
        fp32_export_ok = True
    except Exception as e:
        print(f"ONNX FP32 export failed for {backbone_name}: {type(e).__name__}: {e}")
        artifacts.append(_nan_artifact_record(backbone_name, "ONNX FP32"))
        fp32_export_ok = False

    # Dynamic INT8
    dynamic_path = backbone_dir / "model_int8_dynamic.onnx"
    if fp32_export_ok:
        try:
            create_dynamic_int8(fp32_path, dynamic_path)

            session = make_ort_session(dynamic_path)
            preds, labels = ort_predict(session, eval_loader)
            metrics = classification_metrics_from_predictions(labels, preds)
            latency = benchmark_onnx_latency(session, eval_loader)

            artifacts.append({
                "Backbone": backbone_name,
                "Artifact": "ONNX INT8 Dynamic",
                "Path": str(dynamic_path),
                "Size_MB": file_size_mb(dynamic_path),
                **metrics,
                "Mean_Latency_ms": latency["mean_batch_latency_ms"],
                "Median_Latency_ms": latency["median_batch_latency_ms"],
                "P95_Latency_ms": latency["p95_batch_latency_ms"],
                "Latency_STD_ms": latency["std_batch_latency_ms"],
                "Throughput_images_s": latency["throughput_images_s"],
            })
        except Exception as e:
            print(f"ONNX Dynamic INT8 failed for {backbone_name}: {type(e).__name__}: {e}")
            artifacts.append(_nan_artifact_record(backbone_name, "ONNX INT8 Dynamic"))
    else:
        artifacts.append(_nan_artifact_record(backbone_name, "ONNX INT8 Dynamic"))

    # Static INT8
    static_path = backbone_dir / "model_int8_static.onnx"
    if not fp32_export_ok:
        print(f"Skipping Static INT8 for {backbone_name}: FP32 export unavailable")
        artifacts.append(_nan_artifact_record(backbone_name, "ONNX INT8 Static"))
    else:
        try:
            create_static_int8(fp32_path, static_path, calibration_loader)

            session = make_ort_session(static_path)
            preds, labels = ort_predict(session, eval_loader)
            metrics = classification_metrics_from_predictions(labels, preds)
            latency = benchmark_onnx_latency(session, eval_loader)

            artifacts.append({
                "Backbone": backbone_name,
                "Artifact": "ONNX INT8 Static",
                "Path": str(static_path),
                "Size_MB": file_size_mb(static_path),
                **metrics,
                "Mean_Latency_ms": latency["mean_batch_latency_ms"],
                "Median_Latency_ms": latency["median_batch_latency_ms"],
                "P95_Latency_ms": latency["p95_batch_latency_ms"],
                "Latency_STD_ms": latency["std_batch_latency_ms"],
                "Throughput_images_s": latency["throughput_images_s"],
            })

        except Exception as e:
            print(f"Static INT8 failed for {backbone_name}: {type(e).__name__}: {e}")
            print("This artifact will be marked as unavailable rather than silently omitted.")
            artifacts.append(_nan_artifact_record(backbone_name, "ONNX INT8 Static"))

    deployment_results.extend(artifacts)

deployment_df = pd.DataFrame(deployment_results)
deployment_df.to_csv(RESULTS_DIR / "deployment_comparison_raw.csv", index=False)
display(deployment_df.round(4))

## Quantitative PTQ analysis

The most important quantities are not raw accuracy alone.

For each **Backbone × Artifact** pair:

### Accuracy retention

`Accuracy Retention (%) = Artifact Accuracy / PyTorch FP32 Accuracy × 100`

### Accuracy degradation

`ΔAccuracy = Artifact Accuracy − PyTorch FP32 Accuracy`

### F1 degradation

`ΔMacro-F1 = Artifact Macro-F1 − PyTorch FP32 Macro-F1`

### Latency speedup

`Speedup = FP32 Reference Latency / Artifact Latency`

### Model compression

`Compression Ratio = FP32 Model Size / Artifact Size`

A useful deployment conclusion therefore looks like:

> "Backbone X retained Y% of baseline macro-F1 while achieving Z× latency speedup and W× storage compression."

That is substantially stronger than saying simply that INT8 was "faster".

In [ ]:
# Add within-backbone deltas relative to PyTorch FP32.
analysis_rows = []

for backbone, group in deployment_df.groupby("Backbone"):
    ref = group[group["Artifact"] == "PyTorch FP32"].iloc[0]

    for _, row in group.iterrows():
        row = row.copy()
        row["Accuracy_Delta"] = row["Accuracy"] - ref["Accuracy"]
        row["Macro_F1_Delta"] = row["Macro_F1"] - ref["Macro_F1"]
        row["Accuracy_Retention_%"] = (
            row["Accuracy"] / ref["Accuracy"] * 100
            if ref["Accuracy"] > 0 else np.nan
        )
        row["F1_Retention_%"] = (
            row["Macro_F1"] / ref["Macro_F1"] * 100
            if ref["Macro_F1"] > 0 else np.nan
        )
        row["Latency_Speedup"] = (
            ref["Mean_Latency_ms"] / row["Mean_Latency_ms"]
            if pd.notna(row["Mean_Latency_ms"]) and row["Mean_Latency_ms"] > 0
            else np.nan
        )
        row["Compression_Ratio"] = (
            ref["Size_MB"] / row["Size_MB"]
            if pd.notna(row["Size_MB"]) and row["Size_MB"] > 0
            else np.nan
        )
        analysis_rows.append(row)

deployment_analysis_df = pd.DataFrame(analysis_rows)
deployment_analysis_df.to_csv(
    RESULTS_DIR / "deployment_comparison_analysis.csv", index=False
)

display(
    deployment_analysis_df[
        [
            "Backbone", "Artifact", "Accuracy", "Macro_F1",
            "Accuracy_Delta", "Macro_F1_Delta",
            "Accuracy_Retention_%", "F1_Retention_%",
            "Size_MB", "Compression_Ratio",
            "Mean_Latency_ms", "Latency_Speedup",
            "Throughput_images_s"
        ]
    ].round(4)
)

In [ ]:
# Heatmap-like table: accuracy by backbone and deployment artifact
accuracy_table = deployment_df.pivot(
    index="Backbone", columns="Artifact", values="Accuracy"
)
display(accuracy_table.round(4))

fig, ax = plt.subplots(figsize=(12, 6))
accuracy_table.plot(kind="bar", ax=ax)
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1)
ax.set_title("Backbone × Deployment Artifact — Accuracy")
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

# Macro-F1
f1_table = deployment_df.pivot(
    index="Backbone", columns="Artifact", values="Macro_F1"
)
fig, ax = plt.subplots(figsize=(12, 6))
f1_table.plot(kind="bar", ax=ax)
ax.set_ylabel("Macro F1")
ax.set_ylim(0, 1)
ax.set_title("Backbone × Deployment Artifact — Macro F1")
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

# Latency
latency_table = deployment_df.pivot(
    index="Backbone", columns="Artifact", values="Mean_Latency_ms"
)
fig, ax = plt.subplots(figsize=(12, 6))
latency_table.plot(kind="bar", ax=ax)
ax.set_ylabel("Mean batch latency (ms)")
ax.set_title("Backbone × Deployment Artifact — Latency")
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

# Size
size_table = deployment_df.pivot(
    index="Backbone", columns="Artifact", values="Size_MB"
)
fig, ax = plt.subplots(figsize=(12, 6))
size_table.plot(kind="bar", ax=ax)
ax.set_ylabel("Serialized model size (MB)")
ax.set_title("Backbone × Deployment Artifact — Model Size")
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## Part III — Backbone × Quantization Interaction

This is the central research-level comparison.

A backbone may be excellent in FP32 but poor after quantization, or vice versa. Therefore, selecting a deployment architecture from FP32 accuracy alone can be misleading.

The following table provides a compact deployment scorecard for every architecture and artifact.

No arbitrary "overall score" is used by default. Combining heterogeneous metrics into a single weighted score requires a stated application-specific utility function; otherwise the weighting can manufacture a preferred winner.

In [ ]:
scorecard_cols = [
    "Backbone", "Artifact",
    "Accuracy", "Macro_F1", "Weighted_F1",
    "Size_MB", "Mean_Latency_ms",
    "Throughput_images_s",
    "Accuracy_Delta", "Macro_F1_Delta",
    "Latency_Speedup", "Compression_Ratio"
]

display(
    deployment_analysis_df[scorecard_cols]
    .sort_values(["Backbone", "Artifact"])
    .round(4)
)

## Pareto analysis

For deployment, a model is attractive when it simultaneously has:

- high accuracy,
- low latency,
- small serialized size.

A point is **Pareto-dominated** if another artifact is at least as good in all three dimensions and strictly better in at least one.

This avoids pretending that one arbitrary weighted score represents every deployment scenario.

In [ ]:
def pareto_front(df, maximize=("Accuracy",), minimize=("Mean_Latency_ms", "Size_MB")):
    rows = df.reset_index(drop=True)
    efficient = []

    for i, a in rows.iterrows():
        dominated = False
        for j, b in rows.iterrows():
            if i == j:
                continue
            no_worse = True
            strictly_better = False

            for col in maximize:
                if b[col] < a[col]:
                    no_worse = False
                    break
                if b[col] > a[col]:
                    strictly_better = True

            if no_worse:
                for col in minimize:
                    if pd.isna(a[col]) or pd.isna(b[col]):
                        no_worse = False
                        break
                    if b[col] > a[col]:
                        no_worse = False
                        break
                    if b[col] < a[col]:
                        strictly_better = True

            if no_worse and strictly_better:
                dominated = True
                break

        efficient.append(not dominated)
    return rows[efficient]

valid_deployment = deployment_analysis_df.dropna(
    subset=["Accuracy", "Mean_Latency_ms", "Size_MB"]
)

pareto_df = pareto_front(valid_deployment)

display(
    pareto_df[
        ["Backbone", "Artifact", "Accuracy",
         "Mean_Latency_ms", "Size_MB", "Macro_F1"]
    ].round(4)
)

fig, ax = plt.subplots(figsize=(10, 7))
for backbone in valid_deployment["Backbone"].unique():
    subset = valid_deployment[valid_deployment["Backbone"] == backbone]
    ax.scatter(
        subset["Mean_Latency_ms"],
        subset["Accuracy"],
        s=90,
        label=backbone
    )

for _, row in pareto_df.iterrows():
    ax.annotate(
        f"{row['Backbone']} / {row['Artifact']}",
        (row["Mean_Latency_ms"], row["Accuracy"]),
        xytext=(5, 5),
        textcoords="offset points"
    )

ax.set_xlabel("Mean batch latency (ms)")
ax.set_ylabel("Accuracy")
ax.set_title("Pareto Analysis: Accuracy vs Latency")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

## Part IV — Statistical and Reproducibility Checks

A research-grade benchmark should not rely on a single latency measurement.

The notebook therefore reports:
- mean latency,
- median latency,
- standard deviation,
- P95 latency,
- throughput.

For accuracy, use the same held-out evaluation set for every artifact. For a formal paper, the next upgrade should be **multiple independent training seeds** and confidence intervals. A three-epoch single-seed benchmark is useful for engineering comparison, but it is not sufficient evidence for a strong statistical claim about backbone superiority.

In [ ]:
# Per-backbone confusion matrices for the FP32 baselines
for backbone_name, model in trained_models.items():
    metrics = evaluate_model(model.to(device), eval_loader)
    cm = confusion_matrix(metrics["labels"], metrics["preds"])

    fig, ax = plt.subplots(figsize=(8, 7))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names
    )
    disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
    ax.set_title(f"{backbone_name} — FP32 Confusion Matrix")
    plt.tight_layout()
    plt.show()

    # Keep models on CPU between experiments.
    model.cpu()

In [ ]:
# Per-class metrics for each FP32 backbone
per_class_records = []

for backbone_name, model in trained_models.items():
    metrics = evaluate_model(model.to(device), eval_loader)
    report = classification_report(
        metrics["labels"],
        metrics["preds"],
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    for cls in class_names:
        per_class_records.append({
            "Backbone": backbone_name,
            "Class": cls,
            "Precision": report[cls]["precision"],
            "Recall": report[cls]["recall"],
            "F1": report[cls]["f1-score"],
            "Support": report[cls]["support"],
        })
    model.cpu()

per_class_df = pd.DataFrame(per_class_records)
display(per_class_df.round(4))
per_class_df.to_csv(RESULTS_DIR / "per_class_backbone_metrics.csv", index=False)

## Final research interpretation template

The notebook intentionally does **not** hard-code a winner.

The final interpretation should answer four separate questions:

### 1. Best predictive backbone
Which backbone has the strongest accuracy and macro-F1 under the same training protocol?

### 2. Best efficiency backbone
Which backbone provides the best latency/throughput and model-size characteristics?

### 3. Best quantization behavior
For each backbone, which PTQ strategy gives the best efficiency while retaining predictive quality?

### 4. Best deployment trade-off
Which **Backbone × Artifact** combinations lie on the Pareto frontier for the intended deployment constraint?

This separation is important because:

**Backbone quality ≠ deployment quality ≠ quantization robustness.**

A defensible research conclusion should report all three.

In [ ]:
# Automatically generate a concise experimental summary.
best_accuracy = backbone_df.loc[backbone_df["Accuracy"].idxmax()]
best_f1 = backbone_df.loc[backbone_df["Macro_F1"].idxmax()]
fastest = backbone_df.loc[backbone_df["mean_batch_latency_ms"].idxmin()]
smallest = backbone_df.loc[backbone_df["Checkpoint_MB"].idxmin()]

print("=== BACKBONE SUMMARY ===")
print(f"Best accuracy: {best_accuracy['Backbone']} ({best_accuracy['Accuracy']:.4f})")
print(f"Best macro-F1: {best_f1['Backbone']} ({best_f1['Macro_F1']:.4f})")
print(f"Fastest FP32 backbone: {fastest['Backbone']} ({fastest['mean_batch_latency_ms']:.2f} ms/batch)")
print(f"Smallest FP32 checkpoint: {smallest['Backbone']} ({smallest['Checkpoint_MB']:.2f} MB)")

print("\n=== PTQ SUMMARY ===")
for backbone, group in deployment_analysis_df.groupby("Backbone"):
    baseline = group[group["Artifact"] == "PyTorch FP32"].iloc[0]
    candidates = group[group["Artifact"] != "PyTorch FP32"].dropna(
        subset=["Accuracy", "Mean_Latency_ms"]
    )

    if len(candidates):
        fastest_ptq = candidates.loc[candidates["Mean_Latency_ms"].idxmin()]
        best_retention = candidates.loc[candidates["Accuracy_Retention_%"].idxmax()]

        print(
            f"{backbone}: fastest={fastest_ptq['Artifact']} "
            f"({fastest_ptq['Latency_Speedup']:.2f}x speedup), "
            f"best accuracy retention={best_retention['Artifact']} "
            f"({best_retention['Accuracy_Retention_%']:.2f}%)"
        )

print("\nResults saved to:", RESULTS_DIR.resolve())

# Experimental limitations and next steps

For publication-quality evidence, extend this notebook in the following order:

1. **Multiple seeds:** train each backbone with at least 3–5 seeds.
2. **Full test set:** use all test samples rather than the development subset.
3. **Matched training budget:** compare equal epochs and, preferably, equal compute budgets.
4. **Report confidence intervals** for accuracy/F1 and repeated latency measurements.
5. **Hardware-specific deployment benchmarks:** CPU and GPU results should not be mixed.
6. **Calibration sensitivity:** test multiple calibration-set sizes and representative calibration samples.
7. **Operator coverage:** record whether static INT8 actually quantizes the intended layers/operators.
8. **Numerical equivalence:** compare ONNX FP32 against PyTorch FP32 before attributing any change to quantization.
9. **Application constraints:** if this is intended for robotics/edge deployment, add memory budget, power, FPS, and real-time deadline as explicit constraints.

The central experimental contribution remains the joint study of:

> **Backbone architecture × training/transfer learning × deployment format × PTQ strategy**

rather than treating ONNX quantization as an isolated post-processing step.